In [7]:
# ============================================================
# KISHA - CALIBRATION AND MEASUREMENT
# ============================================================

# Discussion:
# This notebook contains only the calibration and measurement
# functions developed by Kisha.
#
# The object detection and image processing modules are not
# imported here because they will be integrated and tested
# later by the testing member.

import cv2

In [8]:
# ============================================================
# 1. CALIBRATION
# ============================================================

# Discussion:
# Calibration determines the relationship between image pixels
# and real-world centimetres.
#
# A reference object with a known physical width is used.
# Its detected pixel width is divided by its actual width
# to calculate the pixels-per-centimetre scale.


def calculate_pixels_per_cm(
    reference_width_pixels,
    reference_width_cm
):
    """
    Calculate pixels per centimetre.

    Formula:
        pixels_per_cm =
        reference_width_pixels / reference_width_cm
    """

    if reference_width_pixels <= 0:
        raise ValueError(
            "Reference width in pixels must be greater than 0."
        )

    if reference_width_cm <= 0:
        raise ValueError(
            "Reference width in cm must be greater than 0."
        )

    pixels_per_cm = (
        reference_width_pixels /
        reference_width_cm
    )

    return pixels_per_cm


def calibrate_from_object(
    reference_object,
    reference_width_cm
):
    """
    Calculate the calibration scale using
    a detected reference object.

    The reference object must contain:
        width_pixels
    """

    reference_width_pixels = (
        reference_object["width_pixels"]
    )

    return calculate_pixels_per_cm(
        reference_width_pixels,
        reference_width_cm
    )

Folder exists: True

Files inside the folder:
.ipynb_checkpoints
compare_detection_methods.py
Course Project_ Object Dimension Measurement .pdf
demo1_read_webcam.py
demo2_regionprop_detect axis_length.py
image_processing.py
praveen_object_detection.py
praveen_regionprops_detection.py
reference
Shaa.ipynb
Student Report Template_ Object Dimension Measurement.docx
__pycache__


In [9]:
# ============================================================
# 2. OBJECT MEASUREMENT
# ============================================================

# Discussion:
# After calibration, the detected object's width and height
# are available in pixels.
#
# This function converts those pixel measurements into
# centimetres using the pixels-per-centimetre scale.


def measure_object(
    object_data,
    pixels_per_cm,
    object_id=1
):

    if pixels_per_cm <= 0:
        raise ValueError(
            "pixels_per_cm must be greater than 0."
        )

    width_pixels = object_data["width_pixels"]
    height_pixels = object_data["height_pixels"]

    width_cm = width_pixels / pixels_per_cm
    height_cm = height_pixels / pixels_per_cm

    return {
        "object_id": object_id,
        "width_pixels": width_pixels,
        "height_pixels": height_pixels,
        "width_cm": width_cm,
        "height_cm": height_cm,
        "area_pixels": object_data.get("area", 0)
    }


def measure_objects(
    objects,
    pixels_per_cm,
    skip_index=None
):

    measurements = []

    for index, obj in enumerate(objects):

        # The reference object can be skipped because
        # it is used for calibration.
        if skip_index is not None and index == skip_index:
            continue

        measurement = measure_object(
            obj,
            pixels_per_cm,
            object_id=len(measurements) + 1
        )

        measurements.append(measurement)

    return measurements

ModuleNotFoundError: No module named 'cv'

In [2]:
# ============================================================
# 3. DISPLAY MEASUREMENTS
# ============================================================

# Discussion:
# This function draws the detected object's bounding box
# and displays its calculated width and height in centimetres.
#
# The reference object can be highlighted separately.


def draw_measurements(
    image,
    objects,
    measurements,
    reference_index=None
):

    result = image.copy()
    measurement_number = 0

    for index, obj in enumerate(objects):

        x = obj["x"]
        y = obj["y"]

        width = obj["width_pixels"]
        height = obj["height_pixels"]

        # Highlight reference object
        if (
            reference_index is not None
            and index == reference_index
        ):

            cv2.rectangle(
                result,
                (x, y),
                (x + width, y + height),
                (255, 255, 0),
                2
            )

            cv2.putText(
                result,
                "Reference",
                (x, max(y - 10, 20)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.6,
                (255, 255, 0),
                2
            )

            continue

        if measurement_number >= len(measurements):
            continue

        measurement = measurements[
            measurement_number
        ]

        measurement_number += 1

        cv2.rectangle(
            result,
            (x, y),
            (x + width, y + height),
            (0, 255, 0),
            2
        )

        label_y = max(y - 10, 20)

        cv2.putText(
            result,
            f"Object {measurement['object_id']}",
            (x, label_y),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.55,
            (0, 255, 0),
            2
        )

        dimension_text = (
            f"W: {measurement['width_cm']:.2f} cm "
            f"H: {measurement['height_cm']:.2f} cm"
        )

        cv2.putText(
            result,
            dimension_text,
            (
                x,
                min(
                    y + height + 20,
                    result.shape[0] - 10
                )
            ),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.50,
            (0, 255, 255),
            2
        )

    return result

In [3]:
# ============================================================
# 4. PRINT MEASUREMENT TABLE
# ============================================================

# Discussion:
# This function prints the final measurement values in a
# simple table. It can be used to check the results and
# prepare the values for the final report.


def print_measurement_table(measurements):

    print("\n" + "-" * 65)
    print("OBJECT MEASUREMENT RESULTS")
    print("-" * 65)

    print(
        f"{'Object':<10}"
        f"{'Width(px)':<12}"
        f"{'Height(px)':<13}"
        f"{'Width(cm)':<12}"
        f"{'Height(cm)':<12}"
    )

    print("-" * 65)

    for m in measurements:

        print(
            f"{m['object_id']:<10}"
            f"{m['width_pixels']:<12}"
            f"{m['height_pixels']:<13}"
            f"{m['width_cm']:<12.2f}"
            f"{m['height_cm']:<12.2f}"
        )

    print("-" * 65)

Calibration Result
------------------
Reference width: 8.56 cm
Reference width in pixels: 856 px
Pixels per cm: 100.00



Object Measurement
------------------
Width: 500 px
Height: 300 px
Width: 5.00 cm
Height: 3.00 cm
